# Real-World Messy Data

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/tutorials/02_real_world_messy_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Ftutorials%2F02_real_world_messy_data.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/tutorials/02_real_world_messy_data.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>

Real datasets are never clean. They arrive with missing values, extreme outliers, and skewed class distributions — and each of these problems is *worse* for quantum encoding than for classical ML.

Why worse? Classical models like neural networks are relatively forgiving: a bit of NaN propagation gets caught by a loss function, and outliers just make gradients noisy. Quantum encoders have no such tolerance — they map raw numbers directly to rotation angles on physical qubits. Feed in a NaN and you get an undefined circuit. Feed in an outlier and you compress 98% of your data into a 2° slice of rotation space.

This tutorial walks through a deliberately messy dataset and fixes each problem step by step, explaining the *quantum-specific* reason each fix matters.

**What you'll learn:**
- Why NaN, outliers, and class imbalance are especially harmful for quantum encoding
- How to apply `Imputer`, `OutlierHandler`, and `ImbalanceHandler` in sequence
- How to verify the data is clean before encoding

**No optional dependencies required.**

In [ ]:
# If running on Colab or Binder, install QuPrep first:
# !pip install quprep

import warnings

import numpy as np

import quprep as qd
from quprep import QuPrepWarning

rng = np.random.default_rng(42)
print(f"quprep {qd.__version__}")

## 1. The messy dataset

We create a dataset with three realistic problems injected:

| Problem | Where | What happens to the encoding |
|---|---|---|
| **Missing values** | 2 cells | `np.sin(NaN) = NaN` — circuit angles become undefined |
| **Extreme outlier** | Feature 0, row 12 | MinMax scaling squashes all 39 normal rows into 2% of the rotation range |
| **Class imbalance** | 90% class 0, 10% class 1 | VQC gradients collapse; model predicts majority class trivially |

In [ ]:
n = 40
X = np.column_stack([
    rng.uniform(1.0, 4.0, n),
    rng.uniform(0.0, 1.0, n),
    rng.uniform(0.5, 2.5, n),
    rng.uniform(0.2, 0.8, n),
])

X[3, 0]  = np.nan    # missing value
X[7, 2]  = np.nan    # missing value
X[12, 0] = 150.0     # extreme outlier
y = np.array([0] * 36 + [1] * 4)  # 90/10 class imbalance

ds = qd.NumpyIngester().load(X, y=y)

print(f"Shape      : {ds.data.shape}")
print(f"NaN count  : {np.isnan(ds.data).sum()}")
print(f"Max value  : {np.nanmax(ds.data):.1f}  ← outlier")
print(f"Class dist : {np.bincount(y)}  (90/10 imbalance)")

## 2. Step 1 — Fix missing values with `Imputer`

NaN propagates through all arithmetic. Encoders call numpy operations — `np.arcsin`, angle scaling, dot products — that produce `NaN` angles when the input contains `NaN`. The resulting circuit is mathematically undefined: a qubit in state `|NaN⟩` has no physical meaning.

`Imputer(strategy="mean")` replaces each `NaN` with the column mean computed from non-missing rows. This is the simplest fix and works well when missing values are rare and the column distribution is roughly symmetric.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    ds_clean = qd.Imputer(strategy="mean").fit_transform(ds)

print(f"NaN count after : {np.isnan(ds_clean.data).sum()}  (was 2)")
print(f"Max value       : {ds_clean.data.max():.1f}  (outlier still present)")

## 3. Step 2 — Clip outliers with `OutlierHandler`

Here's the quantum-specific problem with outliers:

MinMax scaling maps `[min, max] → [0, π]`. With one outlier at `150.0`, the scaler sees a range of `[1.0, 150.0]`. All 39 normal samples — whose values sit between 1 and 5 — get mapped to the first **2.7%** of the rotation range ([0, 0.085 rad]). The 39 circuits are nearly identical. The quantum model is essentially blind.

`OutlierHandler(method="iqr", action="clip")` computes the 1.5×IQR fence for each column and clips values to that boundary. Using `action="clip"` instead of `action="remove"` preserves all rows — important when your dataset is small.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    ds_no_outliers = qd.OutlierHandler(method="iqr", action="clip").fit_transform(ds_clean)

print(f"Max value after : {ds_no_outliers.data.max():.2f}  (was 150.0, now clipped to IQR fence)")
print(f"Rows kept       : {ds_no_outliers.data.shape[0]}  (clip preserves all rows)")

## 4. Step 3 — Balance classes with `ImbalanceHandler`

A 90/10 class split means a variational quantum classifier (VQC) sees 36 majority-class examples before encountering a single minority sample during training. The circuit parameters drift toward predicting the majority class on every input — a degenerate solution that achieves 90% accuracy by ignoring the minority entirely.

**SMOTE** (Synthetic Minority Over-sampling Technique) generates new minority samples by interpolating between real minority examples in feature space. Unlike simple duplication, SMOTE adds variation, which helps the VQC learn decision boundaries rather than just memorising samples.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    ds_balanced = qd.ImbalanceHandler(strategy="smote").fit_transform(ds_no_outliers)

labels = ds_balanced.labels.astype(int)
print(f"Class dist after : {np.bincount(labels)}  (was [36, 4])")
print(f"Total rows       : {ds_balanced.data.shape[0]}  (was 40, SMOTE added synthetic samples)")

## 5. Step 4 — Scale and encode

With the data clean, outlier-free, and balanced, the final step is scaling to `[0, π]` and encoding. The `Pipeline` here handles just normalisation and encoding — the cleaning steps above ran standalone.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    result = qd.Pipeline(
        normalizer=qd.Scaler(strategy="minmax_pi"),
        encoder=qd.AngleEncoder(),
    ).fit_transform(ds_balanced)

print(f"Scaled range : [{result.dataset.data.min():.3f}, {result.dataset.data.max():.3f}]")
print(f"Circuits     : {len(result.encoded)}")

## 6. Verify compatibility and inspect the circuit

In [ ]:
report = qd.check_compatibility(qd.AngleEncoder(), result.dataset)

print(f"Compatible : {report.is_compatible}")
print(f"Warnings   : {len(report.warnings)}")
print(f"Errors     : {len(report.errors)}")
print()
print(qd.draw_ascii(result.encoded[0]))

## Next steps

| I want to... | Go to |
|---|---|
| Auto-generate a pipeline and export to Qiskit | [tutorials/03_end_to_end_with_a_framework](03_end_to_end_with_a_framework.ipynb) |
| See all imbalance strategies (oversample, ADASYN) | [how-to/fix_class_imbalance](../how-to/fix_class_imbalance.ipynb) |
| Run a full data audit before encoding | [how-to/validate_before_encoding](../how-to/validate_before_encoding.ipynb) |
| Detect drift in production data | [how-to/detect_data_drift](../how-to/detect_data_drift.ipynb) |